# 04 · NULLs and data quality

**Dataset:** 1,000 synthetic healthcare claims (January–June 2026). All data is fake.

| Table | Columns |
|---|---|
| `claims` (1,000) | `claim_id`, `patient_id`, `provider_id`, `date_of_service`, `cpt_code`, `billed_amount`, `paid_amount`, `claim_status`, `denial_reason` |
| `patients` (200) | `patient_id`, `patient_name`, `date_of_birth`, `insurance_plan`, `state` |
| `providers` (13) | `provider_id`, `provider_name`, `specialty`, `state` |
| `procedures` (13) | `cpt_code`, `description`, `service_type`, `standard_charge` |

> 🇪🇸 **Para Josué:** hoy toca **NULL** (celda vacía) y **calidad de datos**: revisar que los datos estén completos y tengan sentido **antes** de sacar conclusiones. En una empresa, esto es la mitad del trabajo de un Data Engineer.
> Como pediste, ya no muestro las tablas: arriba tienes la lista de columnas. La primera vez, corre todo con **Run All**. **Guarda con Ctrl + S después de cada ejercicio.**

In [1]:
# Load the CSV files as tables.
import duckdb

duckdb.sql("CREATE OR REPLACE TABLE claims     AS SELECT * FROM '../data/claims.csv'")
duckdb.sql("CREATE OR REPLACE TABLE patients   AS SELECT * FROM '../data/patients.csv'")
duckdb.sql("CREATE OR REPLACE TABLE providers  AS SELECT * FROM '../data/providers.csv'")
duckdb.sql("CREATE OR REPLACE TABLE procedures AS SELECT * FROM '../data/procedures.csv'")
print("Tables loaded: claims, patients, providers, procedures")

Tables loaded: claims, patients, providers, procedures


---
## Example · The `= NULL` trap

> 🇪🇸 **Te recuerdo:** NULL no es un valor, es "no sé". Por eso `= NULL` **nunca** encuentra nada (0 filas), aunque haya celdas vacías.
> Para buscar celdas vacías siempre se usa **`IS NULL`**, y para las llenas, **`IS NOT NULL`**.

In [12]:
duckdb.sql("""
    SELECT COUNT(*) AS rows_found
    FROM claims
    WHERE paid_amount = NULL
""").df()

,rows_found
0,0


In [3]:
duckdb.sql("""
    SELECT COUNT(*) AS rows_found
    FROM claims
    WHERE paid_amount IS NULL
""").df()

,rows_found
0,63


> 🇪🇸 La misma pregunta: con `=` salen 0 y con `IS NULL` salen 63. Solo la segunda es correcta.

## Exercise 1 · Claims without payment

List the claims that have **no** `paid_amount` (empty), oldest first.

Columns: `claim_id`, `date_of_service`, `billed_amount`, `claim_status`.

✔ **63 rows.** The first one is **CL00045** (2026-01-09, $91). Look at `claim_status`: **all of them are Pending**.

> 🇪🇸 Tiene sentido de negocio: un claim pendiente todavía no tiene pago, así que su `paid_amount` está vacío (NULL), no en 0. Un Denied sí tiene 0: ya se sabe que no se pagó.

In [36]:
duckdb.sql("""
    SELECT c.claim_id, c.date_of_service, c.billed_amount, c.claim_status
    FROM claims as c
    WHERE c.paid_amount IS NULL
    ORDER BY c.date_of_service
""").df()

,claim_id,date_of_service,billed_amount,claim_status
0,CL00045,2026-01-09,91,Pending
1,CL00075,2026-01-14,286,Pending
2,CL00080,2026-01-15,165,Pending
3,CL00103,2026-01-19,188,Pending
4,CL00108,2026-01-19,195,Pending
...,...,...,...,...
58,CL00892,2026-06-11,180,Pending
59,CL00898,2026-06-12,314,Pending
60,CL00921,2026-06-16,174,Pending
61,CL00949,2026-06-21,159,Pending


---
## Example · COUNT(*) vs COUNT(column)

> 🇪🇸 **Te recuerdo:** `COUNT(*)` cuenta **filas**. `COUNT(columna)` cuenta solo las celdas **llenas** de esa columna: se salta los NULL. (El 0 sí cuenta, porque 0 es un dato.)

In [5]:
duckdb.sql("""
    SELECT COUNT(*) AS total_rows,
           COUNT(paid_amount) AS rows_with_paid_amount
    FROM claims
""").df()

,total_rows,rows_with_paid_amount
0,1000,937


## Exercise 2 · How complete is each column?

Show **one row** with: the total number of claims, how many have a `paid_amount`, and how many have a `denial_reason`.

Columns: `total_rows`, `with_paid`, `with_reason`.

✔ **1 row:** 1,000 · 937 · 174.

> 🇪🇸 Como el ejemplo, con una columna más. Piensa: ¿por qué `with_reason` es 174? ¿Te suena ese número del tema 2?

In [14]:
duckdb.sql("""
    SELECT 
    COUNT(*) AS total_rows, 
    COUNT(paid_amount) AS with_paid,
    COUNT(denial_reason) AS with_reason
    FROM claims
""").df()

,total_rows,with_paid,with_reason
0,1000,937,174


---
## New: NULLs change averages

`AVG` **skips** NULLs, the same way `COUNT(column)` does. With `COALESCE` you can treat NULL as 0 instead.

> 🇪🇸 **Lo nuevo:** `AVG(paid_amount)` = suma de los pagos ÷ **solo los claims con pago**. Los 63 Pending no entran en la división.
> `AVG(COALESCE(paid_amount, 0))` = los Pending cuentan como 0, así que entran en la división y el promedio **baja**.
> **Te recuerdo:** `COALESCE(columna, 0)` = "si está vacío, pon 0". Es el `SI.ERROR()` o `SI(ESBLANCO())` de Excel.
> Ninguno está "mal": depende de la pregunta. Lo importante es **saber cuál estás calculando** y decirlo en el reporte.

**Example:** the average billed amount (no NULLs here, so both give the same).

In [17]:
duckdb.sql("""
    SELECT ROUND(AVG(billed_amount),2) AS avg_billed,
           ROUND(AVG(COALESCE(billed_amount,0)),2) AS avg_billed_null_as_zero
    FROM claims
""").df()

,avg_billed,avg_billed_null_as_zero
0,425.09,425.09


## Exercise 3 · Average payment, two ways

Show **one row** with the average `paid_amount` calculated in two ways, both rounded to 2 decimals:
- `avg_ignoring_null`: skipping the empty values.
- `avg_null_as_zero`: counting the empty values as 0.

Columns: `avg_ignoring_null`, `avg_null_as_zero`.

✔ **1 row:** 207.86 · 194.76.

> 🇪🇸 Igual que el ejemplo, pero con `paid_amount`. Aquí sí hay NULL (los 63 Pending), por eso los números cambian.

In [18]:
duckdb.sql("""
    SELECT ROUND(AVG(paid_amount),2) AS avg_ignoring_null,
    ROUND(AVG(COALESCE(paid_amount,0)),2) AS avg_null_as_zero
    FROM claims

""").df()

,avg_ignoring_null,avg_null_as_zero
0,207.86,194.76


## Exercise 4 · Billing by provider name, without losing claims

For each **provider name**, show the number of claims and the total billed, from the highest to the lowest total billed.
The claims with a provider that does not exist must **not** be lost: show them as **`Unknown provider`**.

Columns: `provider_name`, `total_claims`, `total_billed`.

✔ **13 rows.** The first one is **Lakeside Imaging** (85 claims, $96,838). The last one is **Unknown provider** (4 claims, $3,305).

> 🇪🇸 Junta tres cosas que ya sabes:
> 1. `LEFT JOIN` de `claims` (izquierda) con `providers`, para no perder los 4 huérfanos.
> 2. `COALESCE(providers.provider_name, 'Unknown provider')`: si el nombre viene vacío, pone ese texto.
> 3. `GROUP BY` por ese nombre.
>
> **Pista:** en el `GROUP BY` puedes repetir el `COALESCE(...)` completo o escribir el nombre que le pusiste con `AS`.

In [32]:
duckdb.sql("""
    SELECT 
    COALESCE(pr.provider_name, 'Unknown provider') AS provider_name,
    COUNT(*) AS total_claims,
    SUM(c.billed_amount) AS total_billed
    FROM claims AS c
    LEFT JOIN providers AS pr
    ON c.provider_id = pr.provider_id
    GROUP BY provider_name
    ORDER BY total_billed DESC

""").df()

,provider_name,total_claims,total_billed
0,Lakeside Imaging,85,96838.0
1,Empire Diagnostic Imaging,87,93578.0
2,Desert Sun Radiology,72,75206.0
3,Golden Coast Heart Center,74,32008.0
4,Northside Cardiology,79,29421.0
5,Harbor Rehab & Therapy,108,18844.0
6,Bayview Physical Therapy,89,15466.0
7,Lone Star Internal Medicine,83,12979.0
8,Riverside Family Practice,86,12921.0
9,Pine Valley Medical Group,85,11885.0


---
## Exercise 5 · Data validation: status vs. denial reason

A claim is **inconsistent** if:
- it is `Denied` but has **no** `denial_reason`, **or**
- it is **not** `Denied` but **has** a `denial_reason`.

Find the inconsistent claims.

Columns: `claim_id`, `claim_status`, `denial_reason`.

✔ **0 rows.** In a data quality check, **0 rows is good news**: it means the data passed the test.

> 🇪🇸 Esto es una **validación**: una consulta que busca errores. Si sale vacía, los datos están bien.
> Necesitas `AND`, `OR` y **paréntesis**, para que SQL sepa qué va junto, igual que en una fórmula de Excel:
> ```sql
> WHERE (condición A AND condición B)
>    OR (condición C AND condición D)
> ```
> "No es Denied" se escribe `claim_status <> 'Denied'` (`<>` = distinto; también vale `!=`).

In [35]:
duckdb.sql("""
    SELECT claim_id, claim_status, denial_reason
FROM claims
WHERE (claim_status = 'Denied' AND denial_reason is NULL)
   OR (denial_reason is NOT NULL AND claim_status <> 'Denied')
""").df()

,claim_id,claim_status,denial_reason


---
## Check your understanding

> 🇪🇸 Responde en el chat, con tus palabras:
> 1. Esta consulta da **164** filas: `SELECT COUNT(*) FROM claims WHERE denial_reason <> 'Duplicate claim'`. Pero hay 1,000 claims y solo 10 son *Duplicate claim*. ¿Por qué no da 990?
> 2. En el ejercicio 3, ¿cuál de los dos promedios usarías para decirle a tu jefe "en promedio nos pagan $X por claim"? ¿Por qué?